# Fock-PARFLM v2.1 — Honest-PPL Sweep Across Checkpoints

**Purpose.** Reconstruct the *honest* (leak-free) PPL trajectory over training and the leak-size trajectory, checkpoint by checkpoint. This decomposes the reported PPL curve (17.99 → 14.59 → 12.88 → 9.50) into **real learning** vs **reverse-channel exploit growth**.

For each checkpoint in `CKPT_NAMES` this notebook runs the two tests from `fock_trained_leak_probe.py`:

1. **Part 1 — future-perturbation probe** (float64): max past-logit shift and mean dNLL of past targets when the future half of the window is swapped. Measures the *far-future* leak channel at that checkpoint's weights.
2. **Part 2 — honest PPL** (`honest_ppl_test`, k targets): scores the same target tokens (a) mid-window with target+future inside the window (the protocol behind every reported PPL) and (b) from a window that ends *before* the target — leak-free by construction. **PPL_B is the honest number; (B − A) in nats is the leak size** (self-visibility + future context combined).

**How to read the result table:**
- Honest PPL_B tracking reported PPL closely at every checkpoint → the numbers are real; the leak is not the story.
- Honest PPL_B flattening (e.g., ~16–18) while reported PPL keeps falling → the recent gains are exploit growth; the case closes as a leak.

**Runtime:** on a high-RAM CPU Colab, roughly 30–60 min per checkpoint at `K=256, BATCH=1` (Part 1 ≈ 10 min + Part 2 ≈ 20–50 min). On a GPU session, minutes per checkpoint with `BATCH=16`.

**Requires:** the repo clone must include `notebooks/conservative_arch/scaleup/debug/fock_trained_leak_probe.py` (commit + push before running in Colab).

In [ ]:
# ============================================================
# Step 1: USER CONFIG — edit to match your Drive layout
# ============================================================

RUN_DIR = 'semsimula_fock_depthcond_vtheta_owt_xi5long_topk16_dt32da16_mh4_dcvt5x8_ob_untied_wsd_e5c_plgate_rep0.05'

_PREFIX = 'fock_dcvt_owt_xi5long_topk16_dt32da16_mh4_dcvt5x8_ob_untied_wsd_e5c_plgate_rep0.05'

# Checkpoints to sweep, in training order (inside RUN_DIR/checkpoints/).
# Reported in-loop PPLs for reference:
#   step75000.pt        -> 17.99 (Phase 3 resume point)
#   step83500_best.pt   -> 14.59
#   step92500_best.pt   -> 12.88
#   step103500_best.pt  ->  9.50
# Add an end-of-Phase-2 checkpoint here too if one is still on Drive.
CKPT_NAMES = [
    f'{_PREFIX}_step75000.pt',
    f'{_PREFIX}_step83500_best.pt',
    f'{_PREFIX}_step92500_best.pt',
    f'{_PREFIX}_step103500_best.pt',
]

VAL_NAME = 'openwebtext_val_2M.npy'

CONTEXT = 512     # MUST be 512 (native trained context)
BATCH   = 1       # 1 on free/high-RAM CPU Colab; 16 on GPU
K       = 256     # honest-PPL target tokens per checkpoint (+/- ~0.16 nats)
N_PAIRS = 4       # Part 1 window pairs per checkpoint
RUN_PART1 = True  # set False to skip the float64 probe (honest PPL only)
DEVICE  = 'cpu'

In [ ]:
# ============================================================
# Step 2: Drive mount + repo clone + Python paths
# ============================================================
import os, sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    REPO_DIR = Path('/content/semsimula-paper')
    if not REPO_DIR.exists():
        !git clone https://github.com/dimitarpg13/semsimula-paper.git /content/semsimula-paper
    else:
        print(f'{REPO_DIR} already exists, pulling latest...')
        !cd /content/semsimula-paper && git pull
else:
    REPO_DIR = Path(os.path.expanduser('~/git/ml/semsimula-paper'))

assert REPO_DIR.exists(), f'Repo not found at {REPO_DIR}'

PARF_DIR         = str(REPO_DIR / 'notebooks' / 'conservative_arch' / 'parf')
CONSERVATIVE_DIR = str(REPO_DIR / 'notebooks' / 'conservative_arch')
DEBUG_DIR        = str(REPO_DIR / 'notebooks' / 'conservative_arch' / 'scaleup' / 'debug')

for p in [PARF_DIR, CONSERVATIVE_DIR, DEBUG_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f'Repo dir: {REPO_DIR}')
print(f'sys.path additions OK.')

# verify the probe script is present in the clone
_probe_file = Path(DEBUG_DIR) / 'fock_trained_leak_probe.py'
assert _probe_file.exists(), (
    f'{_probe_file} not found — commit and push it, then re-run this cell.')
print(f'Probe script found: {_probe_file}')

In [ ]:
# ============================================================
# Step 3: Paths, dummy logfreq, validation data
# ============================================================
import numpy as np
import tempfile

if IN_COLAB:
    GDRIVE_ROOT = Path(f'/content/drive/MyDrive/{RUN_DIR}')
else:
    GDRIVE_ROOT = Path(f'/path/to/local/{RUN_DIR}')  # local override

CKPT_PATHS = [GDRIVE_ROOT / 'checkpoints' / n for n in CKPT_NAMES]
VAL_PATH   = GDRIVE_ROOT / 'data' / VAL_NAME

for p in CKPT_PATHS:
    print(f'{"OK " if p.exists() else "MISSING "} {p.name}')
missing = [p for p in CKPT_PATHS if not p.exists()]
assert not missing, f'Missing checkpoints: {[p.name for p in missing]}'
assert VAL_PATH.exists(), f'Val data not found: {VAL_PATH}'

# Dummy logfreq (trained values come from the checkpoint's state_dict)
LOGFREQ_PATH = Path(tempfile.gettempdir()) / 'dummy_logfreq.npy'
np.save(LOGFREQ_PATH, np.full(50257, -np.log(1.0 / 50257), dtype=np.float32))

val_tokens = np.asarray(np.load(str(VAL_PATH), mmap_mode='r')).reshape(-1)
print(f'\nValidation set: {val_tokens.shape[0]:,} tokens (dtype={val_tokens.dtype})')

In [ ]:
# ============================================================
# Step 4: Build the model ONCE (from the first checkpoint)
# ============================================================
# build_model constructs the exact training config (incl. the depth-
# conditioned Gaussian V_theta and the per-layer reverse_channel_scale
# patch). Subsequent checkpoints of the same run only need
# load_state_dict on this same model instance.
import gc
import torch
from eval_ppl_proper import build_model

print(f'[load] {CKPT_PATHS[0].name}')
_ckpt = torch.load(str(CKPT_PATHS[0]), map_location='cpu')
model = build_model(_ckpt, DEVICE, logfreq_path=str(LOGFREQ_PATH))
_first_reported = _ckpt.get('val_ppl', None)
del _ckpt
gc.collect()

n_params = sum(p.numel() for p in model.parameters())
print(f'[model] parameters: {n_params:,}  device: {DEVICE}')

In [ ]:
# ============================================================
# Step 5: Sweep — probe + honest PPL for every checkpoint
# ============================================================
import json
import math
import time
from datetime import datetime

from fock_trained_leak_probe import probe_trained_leak, honest_ppl_test

if IN_COLAB:
    log_dir = Path(f'/content/drive/MyDrive/{RUN_DIR}/eval_logs')
    log_dir.mkdir(parents=True, exist_ok=True)
    sweep_log = log_dir / 'honest_ppl_sweep.jsonl'
else:
    sweep_log = None

rows = []
for i, (name, path) in enumerate(zip(CKPT_NAMES, CKPT_PATHS)):
    print('\n' + '#' * 70)
    print(f'# [{i + 1}/{len(CKPT_NAMES)}] {name}')
    print('#' * 70)
    t0 = time.time()

    # --- load weights into the already-built model ---
    ckpt = torch.load(str(path), map_location='cpu')
    sd = ckpt.get('model_state_dict', ckpt)
    reported_ppl = ckpt.get('val_ppl', None)
    step = ckpt.get('step', None)
    missing, unexpected = model.load_state_dict(sd, strict=False)
    unexpected = [k for k in unexpected if k != 'reverse_warmup_step']
    if missing or unexpected:
        print(f'  [warn] missing={missing[:5]} unexpected={unexpected[:5]}')
    del ckpt, sd
    gc.collect()
    model.eval()
    print(f'  step={step}  reported in-loop PPL={reported_ppl}')

    # --- Part 1: future-perturbation probe (float64) ---
    probe_res = None
    if RUN_PART1:
        probe_res = probe_trained_leak(model, val_tokens, device=DEVICE,
                                       context=CONTEXT, n_pairs=N_PAIRS)

    # --- Part 2: honest PPL ---
    honest_res = honest_ppl_test(model, val_tokens, k=K, context=CONTEXT,
                                 batch=BATCH, device=DEVICE)

    row = {
        'timestamp': datetime.now().isoformat(),
        'checkpoint': name,
        'step': step,
        'reported_inloop_ppl': reported_ppl,
        'k': K,
        'ppl_mid_window_standard': round(honest_res['ppl_mid_window'], 4),
        'ppl_last_pos_leak_free': round(honest_res['ppl_last_pos'], 4),
        'paired_diff_nats': round(honest_res['paired_diff_nats'], 6),
        'paired_diff_se': round(honest_res['paired_diff_se'], 6),
        'probe_max_dlogit_past':
            probe_res['max_dlogit_past'] if probe_res else None,
        'probe_mean_dnll_past_nats':
            round(probe_res['mean_dnll_past'], 6) if probe_res else None,
        'wall_time_sec': round(time.time() - t0, 1),
    }
    rows.append(row)

    if sweep_log is not None:  # persist immediately, survives disconnects
        with open(sweep_log, 'a') as f:
            f.write(json.dumps(row) + '\n')
        print(f'  saved -> {sweep_log}')

print('\nSweep complete.')

In [ ]:
# ============================================================
# Step 6: Summary table — the honest vs reported trajectory
# ============================================================
hdr = (f'{"step":>8} {"reported":>9} {"std(A)":>8} {"honest(B)":>10} '
       f'{"B-A nats":>10} {"+/-":>7} {"probe dNLL":>11}')
print(hdr)
print('-' * len(hdr))
for r in rows:
    rep = f'{r["reported_inloop_ppl"]:.2f}' if r['reported_inloop_ppl'] else '—'
    pd = (f'{r["probe_mean_dnll_past_nats"]:+.4f}'
          if r['probe_mean_dnll_past_nats'] is not None else '—')
    print(f'{r["step"] or "?":>8} {rep:>9} '
          f'{r["ppl_mid_window_standard"]:>8.2f} '
          f'{r["ppl_last_pos_leak_free"]:>10.2f} '
          f'{r["paired_diff_nats"]:>+10.4f} {r["paired_diff_se"]:>7.4f} '
          f'{pd:>11}')

print("""
Interpretation:
  honest(B) falls in step with reported  -> gains are real learning
  honest(B) flattens while reported falls -> exploit growth; leak confirmed;
                                             honest(B) is the model's real PPL
  B-A ~ 0 at every checkpoint             -> full-window protocol is honest;
                                             pivot to data-side checks
""")

if IN_COLAB:
    summary_file = log_dir / 'honest_ppl_sweep_summary.txt'
    with open(summary_file, 'w') as f:
        f.write('Fock-PARFLM v2.1 Honest-PPL Sweep\n')
        f.write('=' * 40 + '\n')
        f.write(f'Generated: {datetime.now().isoformat()}\n')
        f.write(f'k={K}  context={CONTEXT}  batch={BATCH}\n\n')
        f.write(hdr + '\n')
        for r in rows:
            f.write(json.dumps(r) + '\n')
    print(f'✅ Summary saved: {summary_file}')